***
# **<center>COURS PYTHON 2IMACS #10</center>**
# ***<center>Initiation au Machine Learning</center>***
# ***<center>Scikit Learn</center>***
# ***<center>Keras Tensorflow</center>***
***

Le **machine learning** consiste à apprendre automatiquement une relation entre des **données d'entrée** et une **sortie cible**.

Dans un cadre **supervisé**, on dispose d'exemples annotés $(X, Y)$, puis on entraîne un modèle pour généraliser à de nouveaux cas.

Dans ce chapitre, on suit le pipeline complet : **préparation des données**, **entraînement**, **prédiction** et **évaluation**.

# 10-1 Donnees tabulaires (Scikit Learn)

## 10-1-1 Import et apercu des donnees

In [ ]:
import pandas as pd
import numpy as np
data = pd.read_csv('fichiers_cours/ia/heart.csv')

In [ ]:
data.shape

In [ ]:
data.head()

In [ ]:
data.describe()

## 10-1-2 Formulation du probleme (X, Y)

Un **modèle** d'apprentissage est une fonction paramétrée qui approxime la relation entre les **caractéristiques** $X$ et la **cible** $Y$.

Après l'entraînement d'un jeu de données, il est utilisé pour produire des **prédictions** sur de nouveaux exemples.

Pour prédire si un patient souffre d'une maladie cardiaque (`heart.csv`), nous séparons les données en deux parties **X** et **Y** :


| Concept | Nom en Machine Learning | Exemples de colonnes | Rôle |
| :--- | :--- | :--- | :--- |
| **$X$** | **Caractéristiques** (*Features*) | Âge, Sexe, Cholestérol, Tension... | Ce que le modèle observe pour décider. |
| **$Y$** | **Cible** (*Target*) | Maladie (0 ou 1) | Ce que le modèle doit prédire. |


Voici comment le modèle traite un patient pour arriver à une décision :

```text
      CARACTÉRISTIQUES (X)                        PRÉDICTION (Y)
   +-----------------------+              +---------------------------+
   |  Âge : 63             |              |                           |
   |  Sexe : Homme (1)     |    MODÈLE    |      RÉSULTAT PROBABLE :  |
   |  Cholestérol : 233    | ---------->  |     [ MALADIE DÉTECTÉE ]  |
   |  Tension : 145        |   (Calculs)  |                           |
   +-----------------------+              +---------------------------+
               ^                                        ^
               |                                        |
        Données d'entrée                        Sortie du modèle
      (Connues du médecin)                     (Aide au diagnostic)
```

L'objectif de l'entraînement est de trouver une fonction $f$ la plus précise possible telle que :

$$Y \approx f(X)$$

Le type de tâche dépend de la nature de la cible $Y$ :

* **Classification** : Si $Y$ est une **catégorie** ou une classe discrète (ex: Malade / Sain, Chat / Chien).
* **Régression** : Si $Y$ est un **nombre continu** ou une valeur quantitative (ex: Taux de sucre, Prix d'une maison).

## 10-1-3 Traitement des donnees categorielles

Le jeu de données contient des variables **catégorielles** (par exemple `sex`, `cp`, `thal`).

Pour éviter d'introduire un ordre artificiel entre catégories, on utilise un encodage **one-hot** : chaque modalité devient une colonne binaire.

Exemple pour `sex` :
- `sex_F = 1` et `sex_M = 0` si le patient est **féminin**
- `sex_F = 0` et `sex_M = 1` si le patient est **masculin**

In [ ]:
# Remplacer les valeurs sex 0,1 par F,M avant l'encodage
data_prep = data.copy()
data_prep['sex'] = data_prep['sex'].map({0: 'F', 1: 'M'})

# One-hot encoding
data_encoded = pd.get_dummies(data_prep, columns=['sex','cp','fbs', 'restecg','exang', 'slope', 'thal'])

# Afficher les premières lignes du DataFrame après l'encodage
data_encoded.head()

## 10-1-4 Separation caracteristiques/cible et train/test

Après séparation de $X$ et $Y$, on découpe les données en deux sous-ensembles :
- un jeu d'**entraînement** pour ajuster les paramètres du modèle
- un jeu de **test** pour mesurer les performances sur des données non vues

In [ ]:
# X contient toutes les caractéristiques (axis = 1 :colonnes) du dataset 
#à l'exception de la variable cible ('target')
X = data_encoded.drop('target', axis=1)
# Extrait la variable cible , c'est à dire à prédire
Y = data_encoded['target']

In [ ]:
print('Dimension des X: ',X.shape)
print('Dimension des Y: ',Y.shape)

La séparation **train/test** permet d'estimer la capacité de **généralisation** du modèle.

Le modèle apprend sur $(X_{train}, Y_{train})$, puis on compare ses prédictions sur $X_{test}$ aux valeurs réelles $Y_{test}$.

In [ ]:
# On divise les patients de manière aléatoire: 80% train et 20% test
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
print('Dimension des X_train: ',X_train.shape)
print('Dimension des Y_train: ',Y_train.shape)
print('Dimension des X_test: ',X_test.shape)
print('Dimension des Y_test: ',Y_test.shape)

## 10-1-5 Normalisation des donnees

Les **échelles** des variables sont différentes (par exemple `chol` peut être élevé alors que `ca` reste faible).

On applique une **standardisation** pour centrer les variables (moyenne proche de 0) et les mettre à échelle comparable (écart-type proche de 1).

Alternative simple possible : une mise à l'échelle par le **maximum** de chaque variable, par exemple
$X_{scaled} = X / X_{max}$.
Dans ce cas, il faut calculer `X_max` sur le **jeu d'entraînement** puis réutiliser ces mêmes maxima pour le jeu de test.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Standard scaler renvoit un tableau numpy. Pour visualiser, convertissons en pandas dataframe
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
# Afficher avec format lisible (pas de notation scientifique)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
X_train_scaled_df.describe()

## 10-1-6 Definition du modele

On utilise une **régression logistique**, modèle linéaire adapté à la **classification binaire**.

Le modèle estime une probabilité d'appartenance à une classe, puis applique un seuil pour produire la classe prédite.

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(random_state=42, max_iter=1000)

## 10-1-7 Entrainement du modele

Pendant l'**entraînement**, le modèle ajuste ses paramètres pour réduire une **fonction de perte** sur les données d'entraînement.

En pratique avec `LogisticRegression`, cette optimisation est gérée automatiquement : l'appel à `fit` réalise les itérations nécessaires jusqu'à convergence (ou `max_iter`).

In [ ]:
# Entraîner le modèle sur les données d'entraînement
model.fit(X_train_scaled, Y_train)

## 10-1-8 Predictions sur le jeu de test

Une fois entraîné, le modèle calcule des **prédictions** sur $X_{test}$ à partir des paramètres appris sur le jeu d'entraînement.

In [ ]:
Y_pred = model.predict(X_test_scaled)

## 10-1-9 Evaluation du modele

On compare les **prédictions** $Y_{pred}$ aux valeurs réelles $Y_{test}$ pour quantifier les **performances** et analyser les erreurs.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
accuracy = accuracy_score(Y_test, Y_pred)
print("Précision du modèle : {:.2f}%".format(accuracy * 100))

La **matrice de confusion** détaille les bonnes classifications et les erreurs par classe.

In [ ]:
conf_matrix = confusion_matrix(Y_test, Y_pred)
print("Matrice de confusion :")
print(conf_matrix)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Matrice de confusion')
plt.xlabel('Prédictions')
plt.ylabel('Vraies étiquettes')
plt.show()

## 10-1-10 Prediction sur un nouveau patient

Considérons un **nouveau patient** avec les caractéristiques suivantes :

- **Âge** : 50 ans
- **Sexe** : Masculin
- **Type de douleur thoracique (`cp`)** : 2
- **Pression artérielle au repos (`trestbps`)** : 140 mm Hg
- **Cholestérol (`chol`)** : 260 mg/dL
- **Glycémie à jeun (`fbs`)** : 0
- **ECG au repos (`restecg`)** : 0
- **Fréquence cardiaque max (`thalach`)** : 165 bpm
- **Angine d'effort (`exang`)** : 0
- **Dépression ST (`oldpeak`)** : 1.2
- **Pente ST (`slope`)** : 1
- **Nombre de vaisseaux (`ca`)** : 1
- **Thal (`thal`)** : 3

Objectif : calculer la **classe prédite** avec le modèle entraîné.

In [ ]:
X_train_scaled_df.columns

In [ ]:
# Regarder les colonnes de X_train
print(X_train.columns)

# Créer X_patientX avec les mêmes colonnes
X_patientX = pd.DataFrame({
    'age': [50],
    'sex_F': [0],
    'sex_M': [1],      # Le patient est un homme (M)
    'cp_0': [0],
    'cp_1': [0],
    'cp_2': [1],       # cp = 2
    'cp_3': [0],
    'fbs_0': [1],
    'fbs_1': [0],
    'restecg_0': [1],
    'restecg_1': [0],
    'restecg_2': [0],
    'exang_0': [1],
    'exang_1': [0],
    'slope_0': [0],
    'slope_1': [0],
    'slope_2': [1],
    'slope_3': [0],
    'thal_0': [0],
    'thal_1': [0],
    'thal_2': [0],
    'thal_3': [1],     # thal = 3
    'trestbps': [140],
    'chol': [260],
    'thalach': [165],
    'oldpeak': [1.2],
    'ca': [1]
})

# S'assurer qu'on a exactement les mêmes colonnes et dans le même ordre
X_patientX = X_patientX[X_train.columns]

Pour un nouvel exemple, il faut appliquer le **même encodage** que sur les données d'entraînement puis réaligner les colonnes sur le schéma de $X_{train}$.

La **normalisation** du patient se fait avec le `StandardScaler` déjà ajusté sur le jeu d'entraînement.

In [ ]:
# Standardiser avec le scaler deja appris sur X_train
X_patientX_scaled = scaler.transform(X_patientX)
print(X_patientX_scaled)

**Prédiction :**

In [ ]:
Prediction_patientX = model.predict(X_patientX_scaled)
print(Prediction_patientX)

Si le modèle prédit `1`, cela signifie que le patient est **susceptible d'avoir une maladie cardiaque**. Si la prédiction est `0`, le modèle estime que le patient est **peu susceptible d'avoir une maladie cardiaque**.

# 10-2 Donnees images (Keras TensorFlow)

## 10-2-1 Chargement du jeu de donnees MNIST

MNIST contient des **images de chiffres manuscrits** (classes de **0 à 9**).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf
import numpy as np
# Charger l'ensemble de données MNIST complet
(X_train, Y_train), (X_test, Y_test) = tf.keras.datasets.mnist.load_data()

# Afficher les dimensions des jeux d'entraînement et de test
print('X_train:', X_train.shape)
print('Y_train:', Y_train.shape)
print('X_test:', X_test.shape)
print('Y_test:', Y_test.shape)

In [ ]:
# Charger les données MNIST
mnist = keras.datasets.mnist
(X, Y), (_, _)= mnist.load_data()

Examinons la **structure** du jeu de données.

In [ ]:
print('X_train',type(X_train))
print('Y_train',type(Y_train))

In [ ]:
print('X_train',X_train.shape)
print('Y_train',Y_train.shape)

## 10-2-2 Normalisation

Avant normalisation, observons les **statistiques des pixels**.

Alternative simple très courante sur MNIST : diviser directement par la valeur maximale d'un pixel (255), soit
$X_{normalized} = X / 255$.
Cette méthode est rapide et souvent suffisante pour un premier modèle.

In [ ]:
# Calculer les statistiques des pixels
min_pixel = np.min(X_train)
max_pixel = np.max(X_train)
mean_pixel = np.mean(X_train)
std_pixel = np.std(X_train)

print('mini:      ',min_pixel)
print('maxi:      ',max_pixel)
print('moyenne:   ',mean_pixel)
print('ecart type ',std_pixel)

In [ ]:
from sklearn.preprocessing import StandardScaler

# Aplatir les images en vecteurs 1D (si elles ne sont pas déjà aplaties)
X_train_flattened = X_train.reshape(len(X_train), -1)
X_test_flattened = X_test.reshape(len(X_test), -1)

# Créer une instance de StandardScaler
scaler = StandardScaler()

# Adapter le scaler aux données d'entraînement et normaliser
X_train_normalized = scaler.fit_transform(X_train_flattened)

# Normaliser les données de test en utilisant le scaler appris sur les données d'entraînement
X_test_normalized = scaler.transform(X_test_flattened)

# Remettre les images dans leur forme originale si nécessaire
X_train_normalized = X_train_normalized.reshape(X_train.shape)
X_test_normalized = X_test_normalized.reshape(X_test.shape)


In [ ]:
# Calculer les statistiques des pixels apres normalisation
min_pixel = np.min(X_train_normalized)
max_pixel = np.max(X_train_normalized)
mean_pixel = np.mean(X_train_normalized)
std_pixel = np.std(X_train_normalized)

print('mini:      ', min_pixel)
print('maxi:      ', max_pixel)
print('moyenne:   ', mean_pixel)
print('ecart type ', std_pixel)

Avec `StandardScaler`, les valeurs ne sont **pas** ramenées dans l'intervalle `[0,1]`. Elles sont **centrées** autour de 0 et **réduites** par l'écart-type, donc certaines peuvent devenir **négatives** ou prendre des valeurs **supérieures à 1** en valeur absolue.

On vérifie donc ici surtout que la **moyenne** est proche de 0 et que l'**écart-type** est proche de 1, plutôt que de chercher un minimum à 0 et un maximum à 1.

In [ ]:
import matplotlib.pyplot as plt
# Générer 16 indices aléatoires entre 0 et 60,000 (taille du jeu de données)
random_indices = np.random.randint(0, len(X_train_normalized), size=16)
# Configuration des sous-graphiques
plt.figure(figsize=(10, 10))
# plt.subplots_adjust(hspace=0.3)

# Boucle pour afficher les images
for i, idx in enumerate(random_indices):
    plt.subplot(4, 4, i+1)
    plt.imshow(X_train_normalized[idx], cmap='gray')
    plt.title('Indice: '+str(idx)+' Y: '+str(Y_train[idx]))
    plt.axis('off')

## 10-2-3 Definition du modele

Le modèle est un **réseau de neurones dense** : couche d'entrée, couches cachées **ReLU**, puis couche de sortie **softmax** sur 10 classes.

In [ ]:
# Nombre de neurones de la premiere couche cachee
hidden1     = 100
# Nombre de neurones de la deuxieme couche cachee
hidden2     = 100

# Construction d'un modele sequence (empilement de couches)
model = keras.Sequential([
    # Forme d'entree des images MNIST (hauteur, largeur)
    keras.layers.Input((28,28)),
    # Transformation de l'image 2D (28x28) en vecteur 1D
    keras.layers.Flatten(),
    # Premiere couche dense avec activation ReLU
    keras.layers.Dense( hidden1, activation='relu'),
    # Deuxieme couche dense avec activation ReLU
    keras.layers.Dense( hidden2, activation='relu'),
    # Couche de sortie a 10 neurones (une probabilité par classe)
    keras.layers.Dense( 10,      activation='softmax')
])

# Choix de l'optimiseur, de la fonction de perte et de la metrique
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
# Affichage du resume de l'architecture du modele
model.summary()

## 10-2-4 Entrainement du modele

On entraîne le modèle avec trois hyperparamètres clés :

- **epochs** : nombre de passes complètes sur les données d'entraînement
- **batch_size** : nombre d'exemples traités avant mise à jour des poids
- **validation_split** : fraction du train utilisée pour suivre la performance hors apprentissage

In [ ]:
history = model.fit(X_train_normalized, Y_train, epochs=16, batch_size=512, validation_split=0.2)

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss during Training and Validation')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy during Training and Validation')

plt.tight_layout()
plt.show()

## 10-2-5 Predictions sur le jeu de test

In [ ]:
Y_pred = model.predict(X_test_normalized)

Affichons les **5 premières sorties** du réseau :

In [ ]:
print(Y_pred[:5])

Chaque prédiction est un vecteur de **probabilités** (une probabilité par classe de **0 à 9**). Vérifions cela sur un exemple.

In [ ]:
print('Etiquette: ',Y_test[1000])
print('Prediction: ',Y_pred[1000])
# Obtenir l'indice de la classe prédite (classe avec la plus haute probabilité)
classe_predite = np.argmax(Y_pred[1000])
print("La classe prédite est :", classe_predite)

On extrait la **classe prédite** avec `np.argmax`, qui renvoie l'indice de probabilité maximale.

In [ ]:
Y_pred_classes = np.argmax(Y_pred, axis=1)

**Vérification :**

In [ ]:
print('Etiquette: ',Y_test[1000])
print('Prediction: ',Y_pred_classes[1000])

## 10-2-6 Evaluation du modele

In [ ]:
# Evaluer le modele sur les donnees de test normalisees
loss, accuracy = model.evaluate(X_test_normalized, Y_test)
print("Loss du modele : {:.2f}".format(loss))
print("Accuracy du modele : {:.2f}%".format(accuracy * 100))

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Faire les predictions sur les donnees de test normalisees
Y_pred = model.predict(X_test_normalized)
Y_pred_classes = np.argmax(Y_pred, axis=1)

# Creer la matrice de confusion
conf_matrix = confusion_matrix(Y_test, Y_pred_classes)

# Affichage de la matrice de confusion
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predictions')
plt.ylabel('Vraies etiquettes')
plt.title('Matrice de confusion')
plt.show()

Observons des **images mal classées** :

In [ ]:
# Trouver les indices des images mal prédites
incorrect_indices = np.where(Y_pred_classes != Y_test)[0]
print(incorrect_indices.shape)

In [ ]:
# Trouver les indices des images mal prédites
incorrect_indices = np.where(Y_pred_classes != Y_test)[0]

# Générer 16 indices aléatoires parmi les indices des images mal prédites
random_incorrect_indices = np.random.choice(incorrect_indices, size=16, replace=False)

# Configuration des sous-graphiques
plt.figure(figsize=(10, 10))

# Boucle pour afficher les images mal prédites avec leurs prédictions
for i, idx in enumerate(random_incorrect_indices):
    plt.subplot(4, 4, i+1)
    plt.imshow(X_test[idx], cmap='gray')
    plt.title('Préd.: '+str(Y_pred_classes[idx])+' / Vrai.: '+str(Y_test[idx]))
    plt.axis('off')

plt.show()


## 10-2-7 Ouverture : PyTorch

PyTorch est une autre bibliotheque majeure pour le deep learning, tres utilisee en recherche et en prototypage rapide.

Specificites souvent mises en avant par rapport a TensorFlow :
- graphes dynamiques natifs, pratiques pour debugger et experimenter
- style de code proche de Python/Numpy, souvent percu comme tres lisible
- ecosysteme recherche tres actif
- TensorFlow garde un avantage historique sur certains usages de production industrielle et de deploiement multi-plateforme